# 04 - Puede un modelo aprender patrones de resistencia, en vez de buscarlos en un diccionario?

El pipeline de `03_pipeline_organizado.ipynb` detecta genes de resistencia (ARG) comparando cada ORF
predicho contra la base curada de CARD: RGI hace, en esencia, una busqueda por alineamiento/homologia
(BLASTP + umbrales de bit-score curados). Es confiable, pero solo encuentra lo que ya esta catalogado.

Este cuaderno prueba una idea distinta: **entrenar un modelo con las secuencias de referencia de CARD
(no con los hits que ya encontro RGI en nuestras muestras)**, para ver si aprende patrones de secuencia
(composicion de k-mers) asociados a resistencia, y luego evaluar ese modelo sobre las secuencias reales
de nuestras muestras -- incluyendo ORFs que RGI nunca marco -- para ver si generaliza mas alla de la
lista de genes conocidos.

Puntos de diseno importantes (se detallan en cada seccion):

- **Entrenamos con CARD, evaluamos con lo real.** Los ARG que RGI ya encontro en nuestras muestras se
  apartan como conjunto de evaluacion "del mundo real" y nunca se usan para entrenar.
- **CARD son genes completos; nuestros ORFs reales son fragmentos** de un ensamblaje metagenomico
  submuestreado. Si no corregimos esto, el modelo podria aprender a distinguir "secuencia completa" vs.
  "fragmento" en lugar de un patron real de resistencia -- por eso fragmentamos las secuencias de CARD a
  una distribucion de longitud similar a la real antes de entrenar.
- **La prueba de generalizacion real es por familia de gen, no por secuencia individual.** Separamos
  familias completas de CARD (p. ej. todas las variantes de `vanW`) entre entrenamiento y prueba, para
  medir si el modelo reconoce algo genuinamente nuevo y no solo memoriza secuencias parecidas.
- Todo el cuaderno usa ML clasico (k-mers + XGBoost / regresion logistica), ya anticipado en
  `environment.yml` (`xgboost`, `imbalanced-learn`, `kmc`). No se necesita GPU y el entrenamiento en si
  toma minutos en un portatil; el unico paso potencialmente lento es la verificacion opcional por BLAST
  remoto al final.

Requiere haber corrido `03_pipeline_organizado.ipynb` sobre al menos algunas muestras (usa
`localDB/card.json`, `work/<run>/genes.fna` y `results/<run>/rgi_<run>.txt`).

## 1. Imports y configuracion

In [1]:
import json
import random
import re
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from Bio import SeqIO
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

random.seed(42)
np.random.seed(42)

DIR_WORK = Path("work")
DIR_RESULTS = Path("results")
CARD_PATH = Path("localDB/card.json")
K = 4  # tamano de k-mer (4^4 = 256 dimensiones; k=5 o 6 quedan como posible mejora, ver seccion 8)

## 2. Conjunto positivo: secuencias de referencia de CARD

Parseamos `localDB/card.json` (la misma base que usa `ensure_card_db()` en el cuaderno 03) y extraemos,
por cada modelo, su secuencia de ADN de referencia junto con la familia de gen, clase de droga y
mecanismo de resistencia segun `ARO_category`. Estas ~6400 secuencias son el **unico** origen de
etiquetas positivas para entrenar -- deliberadamente no usamos los hits que RGI ya encontro en nuestras
muestras.

In [2]:
def cargar_card_positivos(card_path=CARD_PATH):
    data = json.loads(card_path.read_text())
    filas = []
    for model_id, modelo in data.items():
        if not isinstance(modelo, dict):
            continue  # claves de metadata como _version, _timestamp
        secuencias = modelo.get("model_sequences", {}).get("sequence", {})
        categorias = modelo.get("ARO_category", {})
        familia = clase_droga = mecanismo = None
        for cat in categorias.values():
            clase = cat.get("category_aro_class_name")
            if clase == "AMR Gene Family" and familia is None:
                familia = cat.get("category_aro_name")
            elif clase == "Drug Class" and clase_droga is None:
                clase_droga = cat.get("category_aro_name")
            elif clase == "Resistance Mechanism" and mecanismo is None:
                mecanismo = cat.get("category_aro_name")
        for seq in secuencias.values():
            dna = seq.get("dna_sequence", {}).get("sequence")
            if not dna:
                continue
            filas.append({
                "model_id": model_id,
                "aro_name": modelo.get("ARO_name"),
                "familia": familia or "desconocida",
                "clase_droga": clase_droga or "desconocida",
                "mecanismo": mecanismo or "desconocido",
                "secuencia": dna.upper(),
            })
    return pd.DataFrame(filas)


card_df = cargar_card_positivos()
print(f"Secuencias positivas de CARD: {len(card_df)}")
print(f"Familias de genes distintas: {card_df['familia'].nunique()}")
card_df.head()

Secuencias positivas de CARD: 6404
Familias de genes distintas: 470


,model_id,aro_name,familia,clase_droga,mecanismo,secuencia
0,2,CblA-1,CblA beta-lactamase,cephalosporin,antibiotic inactivation,ATGAAAGCATATTTCATCGCCATACTTACCTTATTCACTTGTATAG...
1,4,SHV-52,SHV beta-lactamase,cephalosporin,antibiotic inactivation,ATGCGTTATATTCGCCTGTGTATTATCTCCCTGTTAGCCGCCCTGC...
2,5,dfrF,trimethoprim resistant dihydrofolate reductase...,diaminopyrimidine antibiotic,antibiotic target replacement,ATGATAGGTTTGATTGTTGCGAGGTCAAAGAATAATGTTATAGGCA...
3,7,CTX-M-130,CTX-M beta-lactamase,cephalosporin,antibiotic inactivation,ATGGTGACAAAGAGAGTGCAACGGATGATGTTCGCGGCGGCGGCGT...
4,8,NDM-6,NDM beta-lactamase,carbapenem,antibiotic inactivation,ATGGAATTGCCCAATATTATGCACCCGGTCGCGAAGCTGAGCACCG...


## 3. ORFs reales: separar hits ya conocidos (evaluacion) del resto (pool negativo)

Para cada muestra procesada por el cuaderno 03, `work/<run>/genes.fna` contiene **todos** los ORFs que
predijo Prodigal, y `results/<run>/rgi_<run>.txt` contiene el subconjunto que RGI marco como ARG (el
encabezado FASTA es identico al valor de `ORF_ID` en la tabla de RGI, asi que separarlos es una simple
diferencia de conjuntos).

- Los hits de RGI se apartan como **conjunto de evaluacion del mundo real** (`positivos_reales`): nunca
  se usan para entrenar, solo para ver si el modelo los recupera a partir del patron de secuencia.
- Todo lo demas (~99.8% de los ORFs) es el **pool negativo** de entrenamiento: en una muestra
  metagenomica real, la enorme mayoria de genes no son de resistencia.

In [3]:
def cargar_orfs_muestra(run):
    fna = DIR_WORK / run / "genes.fna"
    rgi_txt = DIR_RESULTS / run / f"rgi_{run}.txt"
    if not fna.exists():
        return pd.DataFrame()
    hits_rgi = set()
    if rgi_txt.exists():
        hits_rgi = set(pd.read_csv(rgi_txt, sep="\t")["ORF_ID"])
    filas = [
        {
            "run": run,
            "orf_id": rec.description,
            "secuencia": str(rec.seq).upper(),
            "es_hit_rgi": rec.description in hits_rgi,
        }
        for rec in SeqIO.parse(fna, "fasta")
    ]
    return pd.DataFrame(filas)


runs = sorted(p.name for p in DIR_WORK.iterdir() if (p / "genes.fna").exists())
orfs_df = pd.concat([cargar_orfs_muestra(r) for r in runs], ignore_index=True)

positivos_reales = orfs_df[orfs_df["es_hit_rgi"]].reset_index(drop=True)
pool_negativo = orfs_df[~orfs_df["es_hit_rgi"]].reset_index(drop=True)

print(f"ORFs totales: {len(orfs_df)} en {len(runs)} muestras")
print(f"  Hits RGI (evaluacion real, NO se entrena con ellos): {len(positivos_reales)}")
print(f"  Pool negativo para entrenamiento: {len(pool_negativo)}")

ORFs totales: 123270 en 12 muestras
  Hits RGI (evaluacion real, NO se entrena con ellos): 180
  Pool negativo para entrenamiento: 123090


## 4. Fragmentar CARD para igualar la distribucion de longitud real

Las secuencias de referencia de CARD son genes completos (mediana ~876 pb); nuestros ORFs reales son
fragmentos de un ensamblaje submuestreado (mediana bastante menor, y muchos truncados en el borde de un
contig). Entrenar solo con genes completos de CARD dejaria una senal facil de explotar: "esta secuencia
es larga y completa" en vez de un patron de composicion real. Por eso generamos, ademas de la secuencia
completa, varios recortes aleatorios de cada secuencia de CARD con longitudes muestreadas de la
distribucion real observada en `pool_negativo`.

In [4]:
longitudes_reales = pool_negativo["secuencia"].str.len()
print(longitudes_reales.describe())


def fragmentar(secuencia, longitud, rng):
    n = len(secuencia)
    if longitud >= n:
        return secuencia
    inicio = rng.randrange(0, n - longitud + 1)
    return secuencia[inicio:inicio + longitud]


def construir_positivos_entrenamiento(card_df, longitudes_objetivo, fragmentos_por_secuencia=3, semilla=42):
    rng = random.Random(semilla)
    longitudes_objetivo = longitudes_objetivo.to_numpy()
    filas = []
    for _, fila in card_df.iterrows():
        filas.append(fila.to_dict())  # version completa: al modelo tambien le sirve ver el gen entero
        for _ in range(fragmentos_por_secuencia):
            longitud = int(rng.choice(longitudes_objetivo))
            frag = fragmentar(fila["secuencia"], longitud, rng)
            if len(frag) < 20:
                continue
            nueva = fila.to_dict()
            nueva["secuencia"] = frag
            filas.append(nueva)
    return pd.DataFrame(filas)


card_entrenamiento = construir_positivos_entrenamiento(card_df, longitudes_reales)
print(f"Positivos de entrenamiento tras fragmentar: {len(card_entrenamiento)} "
      f"(de {len(card_df)} secuencias originales de CARD)")

count    123090.000000
mean        426.519620
std         326.419615
min          60.000000
25%         231.000000
50%         357.000000
75%         513.000000
max        9195.000000
Name: secuencia, dtype: float64
Positivos de entrenamiento tras fragmentar: 25616 (de 6404 secuencias originales de CARD)


## 5. Features: frecuencia de k-mers de nucleotidos

Usamos composicion de k-mers (k=4 por defecto, "firma genomica" tetranucleotidica) en vez de alineamiento
contra una referencia: es una tecnica reference-free establecida (usada p. ej. para binning taxonomico) y
es exactamente el tipo de senal "de patron" que le falta a una busqueda por diccionario. Se calcula
directamente en Python (para secuencias de este tamano no hace falta el binario `kmc` del entorno).

In [5]:
def vocabulario_kmers(k):
    return ["".join(p) for p in product("ACGT", repeat=k)]


def vector_kmer(secuencia, k, vocab_index):
    vec = np.zeros(len(vocab_index), dtype=np.float32)
    secuencia = re.sub(r"[^ACGT]", "", secuencia)
    total = 0
    for i in range(len(secuencia) - k + 1):
        idx = vocab_index.get(secuencia[i:i + k])
        if idx is not None:
            vec[idx] += 1
            total += 1
    if total:
        vec /= total
    return vec


def matriz_kmers(secuencias, k=K):
    vocab_index = {kmer: i for i, kmer in enumerate(vocabulario_kmers(k))}
    return np.vstack([vector_kmer(s, k, vocab_index) for s in secuencias])


negativos_entrenamiento = pool_negativo.sample(
    n=min(len(pool_negativo), len(card_entrenamiento) * 3), random_state=42
)

X_pos = matriz_kmers(card_entrenamiento["secuencia"])
X_neg = matriz_kmers(negativos_entrenamiento["secuencia"])

X = np.vstack([X_pos, X_neg])
y = np.concatenate([np.ones(len(X_pos)), np.zeros(len(X_neg))])
familias = np.concatenate([
    card_entrenamiento["familia"].to_numpy(),
    np.array(["(negativo)"] * len(X_neg)),
])

print(f"Dataset de entrenamiento: {X.shape}, positivos={int(y.sum())}, negativos={int((y == 0).sum())}")

Dataset de entrenamiento: (102464, 256), positivos=25616, negativos=76848


## 6. Dos particiones de evaluacion

- **Split aleatorio**: reparte secuencias al azar entre train/test. Sirve como chequeo de cordura, pero
  no responde la pregunta de generalizacion (secuencias muy parecidas de la misma familia pueden quedar
  en ambos lados).
- **Split por familia de gen (holdout)**: aparta ~20% de las 522 familias de CARD *completas* para
  prueba -- ninguna variante de esas familias se ve en entrenamiento. Esta es la prueba real de si el
  modelo generaliza mas alla de una lista conocida, en vez de solo memorizar.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

familias_unicas = sorted(set(card_entrenamiento["familia"]))
rng_split = random.Random(7)
rng_split.shuffle(familias_unicas)
n_holdout = max(1, int(0.2 * len(familias_unicas)))
familias_holdout = set(familias_unicas[:n_holdout])

es_holdout = np.isin(familias, list(familias_holdout))
es_positivo = y == 1

test_mask_fam = es_holdout & es_positivo
train_mask_fam = ~test_mask_fam

neg_idx = np.where(y == 0)[0]
neg_test_idx = rng_split.sample(list(neg_idx), k=min(len(neg_idx), int(0.2 * len(neg_idx))))
test_mask_fam[neg_test_idx] = True
train_mask_fam[neg_test_idx] = False

X_train_fam, y_train_fam = X[train_mask_fam], y[train_mask_fam]
X_test_fam, y_test_fam = X[test_mask_fam], y[test_mask_fam]

assert not (set(familias[train_mask_fam & es_positivo]) & familias_holdout), "fuga de familias al train"
print(f"Familias en holdout: {len(familias_holdout)}/{len(familias_unicas)}")
print(f"Split por familia -> train: {X_train_fam.shape}, test: {X_test_fam.shape}")

Familias en holdout: 94/470
Split por familia -> train: (83307, 256), test: (19157, 256)


## 7. Entrenamiento: regresion logistica (baseline) y XGBoost

In [7]:
def entrenar_evaluar(X_train, y_train, X_test, y_test, nombre_split):
    modelos = {
        "regresion logistica": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "xgboost": XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.1,
            scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
            eval_metric="logloss", random_state=42,
        ),
    }
    entrenados = {}
    for nombre_modelo, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        proba = modelo.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)
        print(f"\n=== {nombre_split} -- {nombre_modelo} ===")
        print(classification_report(y_test, pred, digits=3))
        print(f"ROC-AUC: {roc_auc_score(y_test, proba):.3f}  "
              f"AUPRC: {average_precision_score(y_test, proba):.3f}")
        entrenados[nombre_modelo] = modelo
    return entrenados


modelos_random = entrenar_evaluar(X_train, y_train, X_test, y_test, "split aleatorio")
modelos_familia = entrenar_evaluar(X_train_fam, y_train_fam, X_test_fam, y_test_fam, "split por familia (holdout)")


=== split aleatorio -- regresion logistica ===
              precision    recall  f1-score   support

         0.0      0.931     0.879     0.904     15370
         1.0      0.689     0.804     0.742      5123

    accuracy                          0.860     20493
   macro avg      0.810     0.842     0.823     20493
weighted avg      0.870     0.860     0.864     20493

ROC-AUC: 0.919  AUPRC: 0.839

=== split aleatorio -- xgboost ===
              precision    recall  f1-score   support

         0.0      0.965     0.971     0.968     15370
         1.0      0.910     0.895     0.902      5123

    accuracy                          0.952     20493
   macro avg      0.938     0.933     0.935     20493
weighted avg      0.951     0.952     0.952     20493

ROC-AUC: 0.982  AUPRC: 0.965

=== split por familia (holdout) -- regresion logistica ===
              precision    recall  f1-score   support

         0.0      0.893     0.887     0.890     15369
         1.0      0.553     0.568  

## 8. Aplicar el modelo a las muestras reales y comparar contra RGI

Entrenamos un modelo final con **todas** las secuencias positivas construidas a partir de CARD (todas las
familias, no solo el 80% del holdout) para maximizar la senal de entrenamiento, y lo aplicamos sobre
**todos** los ORFs reales de nuestras 12 muestras (no solo los que RGI ya marco).

Dos numeros importan aqui:
- **Recall sobre los hits ya conocidos de RGI**: si el modelo no recupera la mayoria de estos, no aprendio
  nada util y no vale la pena mirar el resto.
- **Candidatos nuevos**: ORFs que el modelo marca como probable ARG pero que RGI *no* marco. Este es el
  resultado que realmente responde la pregunta original -- pero son candidatos sin verificar, no ARG
  confirmados (ver seccion 9).

In [8]:
modelo_final = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    eval_metric="logloss", random_state=42,
)
modelo_final.fit(X, y)

X_real = matriz_kmers(orfs_df["secuencia"])
orfs_df["proba_arg"] = modelo_final.predict_proba(X_real)[:, 1]

UMBRAL = 0.5
recall_conocidos = orfs_df.loc[orfs_df["es_hit_rgi"], "proba_arg"].ge(UMBRAL).mean()
print(f"Recall sobre los {orfs_df['es_hit_rgi'].sum()} ARG ya conocidos (RGI): {recall_conocidos:.1%}")

candidatos_nuevos = (
    orfs_df[(~orfs_df["es_hit_rgi"]) & (orfs_df["proba_arg"] >= UMBRAL)]
    .sort_values("proba_arg", ascending=False)
)
print(f"Candidatos NO catalogados por RGI, marcados como posible ARG por el modelo: {len(candidatos_nuevos)} "
      f"(de {len(pool_negativo)} ORFs no-ARG)")
candidatos_nuevos[["run", "orf_id", "proba_arg"]].head(20)

Recall sobre los 180 ARG ya conocidos (RGI): 35.6%
Candidatos NO catalogados por RGI, marcados como posible ARG por el modelo: 2874 (de 123090 ORFs no-ARG)


,run,orf_id,proba_arg
64964,ERR1135386,k119_2657_1 # 3 # 212 # 1 # ID=1792_1;partial=...,0.987072
103887,ERR1135451,k119_1842_1 # 1 # 462 # 1 # ID=1980_1;partial=...,0.986084
91443,ERR1135436,k119_6574_1 # 2 # 391 # 1 # ID=7367_1;partial=...,0.980905
50104,ERR1135222,k99_3827_7 # 4242 # 4784 # 1 # ID=11941_7;part...,0.979248
93851,ERR1135438,k119_1138_1 # 1 # 135 # -1 # ID=1133_1;partial...,0.978272
28979,ERR1135202,k99_4280_1 # 903 # 980 # 1 # ID=4331_1;partial...,0.974831
118273,ERR1135463,k119_6549_2 # 310 # 459 # 1 # ID=6357_2;partia...,0.972699
27469,ERR1135202,k99_4007_2 # 413 # 481 # 1 # ID=3130_2;partial...,0.970352
21049,ERR1135194,k99_623_2 # 479 # 544 # -1 # ID=6285_2;partial...,0.969331
12988,ERR1135194,k99_6102_1 # 343 # 417 # -1 # ID=383_1;partial...,0.965817


## 9. (Opcional, lento) Verificar por BLAST remoto los candidatos con mayor probabilidad

Reutiliza `blast_remoto()` del cuaderno 03. **Cada consulta puede tardar 1-5 minutos** y NCBI limita la
tasa de consultas remotas -- por eso solo verificamos un puñado de los candidatos con mayor probabilidad,
no toda la lista. Un "sin hits" no descarta que sea un ARG real (podria ser genuinamente nuevo); un hit
contra un gen conocido no relacionado con resistencia sugiere que fue una falsa alarma del modelo.

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML
import time


def blast_remoto(fasta_str, program="blastn", db="nt", hitlist=1, max_reintentos=3):
    ultimo = None
    for intento in range(1, max_reintentos + 1):
        try:
            handle = NCBIWWW.qblast(program, db, fasta_str, hitlist_size=hitlist, megablast=True)
            return list(NCBIXML.parse(handle))
        except Exception as e:
            ultimo = e
            print(f"  Intento {intento}/{max_reintentos} -> {type(e).__name__}: {e}")
            if intento < max_reintentos:
                time.sleep(20)
    raise RuntimeError(f"BLAST fallo tras {max_reintentos} intentos") from ultimo


N_VERIFICAR = 5
for _, fila in candidatos_nuevos.head(N_VERIFICAR).iterrows():
    print(f"\n--- {fila['orf_id']} (run={fila['run']}, proba={fila['proba_arg']:.2f}) ---")
    try:
        registros = blast_remoto(f">{fila['orf_id']}\n{fila['secuencia']}\n")
        rec = registros[0]
        if rec.alignments:
            top = rec.alignments[0]
            print(f"  Mejor hit: {top.hit_def[:90]}")
        else:
            print("  Sin hits en BLAST (posible secuencia genuinamente nueva, o ruido del modelo)")
    except Exception as e:
        print(f"  Fallo BLAST: {e}")

## 10. Discusion y limitaciones

- **Escala real todavia chica**: solo 12 muestras procesadas y 180 ARG confirmados por RGI para evaluar
  contra el mundo real. Suficiente para un primer piloto, no para conclusiones fuertes; correr el
  cuaderno 03 sobre mas `RUNS` de ENA daria un conjunto de evaluacion mas confiable.
- **Cambio de dominio CARD vs. metagenoma**: mitigamos la diferencia de longitud fragmentando, pero
  restan otras diferencias (calidad de ensamblaje, errores de secuenciacion, ORFs parciales en el borde
  de un contig) que CARD no tiene. Si el recall sobre hits conocidos es bajo, esta es la primera
  sospechosa.
- **Los "candidatos nuevos" no estan verificados**: sin una referencia externa (BLAST, revision experta,
  o idealmente validacion de laboratorio) no podemos afirmar que sean ARG reales, solo que el modelo los
  considera composicionalmente parecidos a los de CARD.
- **Si este baseline rinde mal**: siguientes pasos razonables, en orden de esfuerzo creciente, son
  probar k=5/6, features a nivel de proteina (k-mers de aminoacidos sobre `genes.faa`) en vez de ADN, y
  como ultimo recurso una CNN pequena sobre la secuencia one-hot -- pero dado que el entorno ya trae
  `xgboost`/`imbalanced-learn`/`kmc`, ese no deberia ser el primer lugar al que saltar.